In [ ]:
# ============================================================
# STEP 1 -- CONFIG (Growth Strategy Research)
# ============================================================
# Mines Reddit for real founder growth stories:
#   - What channels worked (SEO, cold email, communities, etc.)
#   - What tactics they used (free tier, lifetime deals, etc.)
#   - What metrics they hit (MRR, users, conversion rates)
#   - What failed and why
#
# NOT interested in: free apps, side projects with no revenue,
# generic advice, "just build something people want" posts.
# We want specifics from people who actually grew paid products.
# ============================================================

import os
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")

# ── Reddit scraping ──────────────────────────────────────────
SUBREDDITS = [
    # Primary (founders sharing real numbers)
    "SaaS", "indiehackers", "microsaas", "startups",
    "Entrepreneur", "SideProject",
    "EntrepreneurRideAlong",

    # Growth-specific
    "growthacking", "digital_marketing", "SEO",
    "PPC", "content_marketing", "emailmarketing",
    "AppBusiness",

    # Dev communities (where builders share launches)
    "webdev", "reactjs", "nextjs", "selfhosted",
    "iOSProgramming", "SwiftUI", "androiddev",

    # Product
    "ProductManagement", "userexperience",
]

LOOKBACK_DAYS   = 7
PAGES_PER_SUB   = 10
REDDIT_SLEEP    = 2.5
FETCH_COMMENTS  = True       # comments often contain the real tactics
TOP_COMMENTS_N  = 5

# ── Growth signal keywords (Step 5) ──────────────────────────
# Post must contain >= 1 to survive the gate filter.
GATE_KEYWORDS = [
    # Revenue / traction signals
    "mrr", "arr", "revenue", "paying users", "paying customers",
    "first sale", "first customer", "first 100", "first 1000",
    "subscribers", "conversion", "churn",
    "$1k", "$5k", "$10k", "$50k", "$100k",
    "ramen profitable", "profitable",

    # Growth tactics
    "growth strategy", "how i grew", "how we grew",
    "how i got", "how we got", "what worked",
    "acquisition channel", "marketing channel",
    "cold email", "cold outreach", "seo", "content marketing",
    "product hunt", "hacker news", "launch",
    "lifetime deal", "appsumo", "ltd",
    "free trial", "freemium", "pricing",
    "referral", "word of mouth", "viral",
    "beta users", "waitlist", "early adopters",

    # Milestone posts
    "hit $", "reached $", "crossed $",
    "from 0 to", "from zero to",
    "month 1", "month 3", "month 6", "month 12",
    "year 1", "year 2",
    "grew to", "scaled to",
    "users in", "customers in",
]

# ── Growth tactic weights (for scoring) ───────────────────────
GROWTH_SIGNALS = {
    # Strong: specific numbers = real experience
    "mrr": 25, "arr": 25, "revenue": 15,
    "paying users": 25, "paying customers": 25,
    "conversion rate": 20, "churn rate": 20,
    "ltv": 20, "cac": 20, "arpu": 20,

    # Tactic mentions
    "cold email": 15, "cold outreach": 15,
    "seo": 12, "content marketing": 12,
    "product hunt": 15, "hacker news": 15,
    "lifetime deal": 15, "appsumo": 15,
    "free trial": 12, "freemium": 12,
    "referral program": 15, "affiliate": 12,
    "beta launch": 12, "waitlist": 12,
    "community": 10, "reddit": 8,
    "twitter": 8, "linkedin": 10,
    "facebook ads": 12, "google ads": 12,
    "influencer": 10, "partnership": 12,

    # Milestone language (shows real progress)
    "from 0 to": 20, "from zero to": 20,
    "hit $": 18, "reached $": 18, "crossed $": 18,
    "grew to": 15, "scaled to": 15,

    # Specificity signals
    "here's what worked": 20, "what actually worked": 22,
    "what didn't work": 18, "what failed": 18,
    "step by step": 15, "breakdown": 12,
    "timeline": 12, "month by month": 15,
    "exact": 15, "specific": 8,
    "numbers": 10, "metrics": 12, "data": 8,
}

MIN_GROWTH_SCORE = 8

# ── Relevance penalties (things we DON'T want) ───────────────
EXCLUDE_SIGNALS = {
    # Free / no-revenue projects
    "free app": -20, "open source": -15,
    "no revenue": -15, "not monetized": -10,
    "just for fun": -15, "hobby project": -15,

    # Generic advice (no specifics)
    "you should": -8, "in my opinion": -5,
    "i think the best way": -8,

    # Job seeking
    "looking for a job": -20, "hire me": -20,
    "portfolio": -10,
}

# ── Topic exclusions ──────────────────────────────────────────
EXCLUDE_TOPICS = [
    # Non-software
    "restaurant", "food truck", "cafe", "bakery",
    "real estate", "rental", "landlord",
    "crypto", "nft", "blockchain", "web3",
    "dropship", "print on demand", "amazon fba",
    "fitness", "gym", "coaching",

    # Emotional / non-tactical
    "burned out", "burnout", "mental health",
    "i quit", "giving up", "impostor syndrome",

    # Free tools / no business model
    "completely free", "100% free", "no monetization",
]

EXCLUDE_TITLE_PATTERNS = [
    r"^ama\b",
    r"\bhiring\b",
    r"\bjob posting\b",
    r"\bweekly thread\b",
    r"\bmonthly thread\b",
    r"\bfeedback friday\b",
    r"\blooking for co-?founder\b",
]

# ── Quality filters ──────────────────────────────────────────
MIN_UPVOTES        = 3      # posts need some validation
MIN_COMMENTS       = 2      # discussion = useful content
MIN_BODY_LENGTH    = 200    # short posts rarely have tactics

# ── Grouping ─────────────────────────────────────────────────
GROUP_SIMILARITY   = 0.30   # higher than pain finder (growth posts are more varied)
MERGE_THRESHOLD    = 0.55

# ── File naming ──────────────────────────────────────────────
def csv_name(step_num, label):
    return f"growth_step{step_num}_{label}_{RUN_ID}.csv"

print(f"Config loaded (Growth Research). Run ID: {RUN_ID}")
print(f"  {len(SUBREDDITS)} subreddits, {LOOKBACK_DAYS}d lookback")
print(f"  Files: growth_step<N>_<label>_{RUN_ID}.csv")

In [ ]:
# ============================================================
# STEP 2 -- INSTALL + VERIFY
# ============================================================

import json, time, re, requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from tqdm.auto import tqdm
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("[OK] All imports successful")

In [ ]:
# ============================================================
# STEP 3 -- SCRAPE REDDIT (same infra, different subs)
# ============================================================
# Same scraper as the pain finder. Fetches posts + top comments.
# Saves: growth_step3_raw_{RUN_ID}.csv
# ============================================================

STEP3_CSV = csv_name(3, "raw")

headers = {"User-Agent": "SaaSGrowthResearch/1.0 (academic)"}
cutoff = time.time() - (LOOKBACK_DAYS * 86400)

print(f"Scraping {len(SUBREDDITS)} subs, last {LOOKBACK_DAYS}d")
print(f"Comments: {'ON' if FETCH_COMMENTS else 'OFF'}\n")

seen = set()
rows = []

def fetch_top_comments(permalink, n=TOP_COMMENTS_N):
    url = f"https://www.reddit.com{permalink}.json?limit={n+5}&sort=top&raw_json=1"
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            return []
        data = r.json()
        if len(data) < 2:
            return []
        comments = data[1].get("data", {}).get("children", [])
        results = []
        for c in comments:
            if c.get("kind") != "t1":
                continue
            cd = c["data"]
            body = (cd.get("body") or "").strip()
            if not body or body in ("[deleted]", "[removed]"):
                continue
            results.append({"text": body[:500], "score": cd.get("score", 0)})
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:n]
    except Exception:
        return []

for sub in tqdm(SUBREDDITS, desc="Subreddits"):
    after_token = None
    sub_count = 0

    for page in range(PAGES_PER_SUB):
        params = {"limit": 100, "raw_json": 1}
        if after_token:
            params["after"] = after_token
        try:
            r = requests.get(
                f"https://www.reddit.com/r/{sub}/new.json",
                headers=headers, params=params, timeout=15,
            )
            if r.status_code == 429:
                time.sleep(int(r.headers.get("Retry-After", 15)))
                continue
            if r.status_code != 200:
                break

            data = r.json().get("data", {})
            children = data.get("children", [])
            if not children:
                break

            for child in children:
                d = child["data"]
                if d["created_utc"] < cutoff or d["id"] in seen:
                    continue
                seen.add(d["id"])
                title = (d.get("title") or "").strip()
                if not title:
                    continue
                body = (d.get("selftext") or "").strip()

                comments_text, comments_scores, comment_count = "", "", 0
                if FETCH_COMMENTS and d.get("num_comments", 0) > 0:
                    tc = fetch_top_comments(d["permalink"])
                    if tc:
                        comments_text = " ||| ".join(c["text"] for c in tc)
                        comments_scores = ", ".join(str(c["score"]) for c in tc)
                        comment_count = len(tc)
                    time.sleep(0.5)

                rows.append({
                    "post_id": d["id"], "subreddit": sub,
                    "title": title, "body": body[:5000],
                    "url": f"https://reddit.com{d['permalink']}",
                    "upvotes": d.get("ups", 0),
                    "upvote_ratio": d.get("upvote_ratio", 0.5),
                    "num_comments": d.get("num_comments", 0),
                    "created_utc": d["created_utc"],
                    "post_age_hours": round((time.time() - d["created_utc"]) / 3600, 1),
                    "flair": d.get("link_flair_text") or "",
                    "top_comments_text": comments_text[:5000],
                    "top_comments_scores": comments_scores,
                    "top_comment_count": comment_count,
                })
                sub_count += 1

            after_token = data.get("after")
            if not after_token:
                break
        except Exception as e:
            tqdm.write(f"  r/{sub} p{page}: {type(e).__name__}")
            break
        time.sleep(REDDIT_SLEEP)

    tqdm.write(f"  r/{sub}: {sub_count}")

df_raw = pd.DataFrame(rows)
df_raw.to_csv(STEP3_CSV, index=False)

print(f"\n{'='*55}")
print(f" SCRAPED: {len(df_raw):,} posts --> {STEP3_CSV}")
print(f"{'='*55}")
if not df_raw.empty:
    for sub, n in df_raw["subreddit"].value_counts().items():
        print(f"  r/{sub:25s} {n:4d}")

In [ ]:
# ============================================================
# STEP 4 -- CLEAN TEXT
# ============================================================
# Same cleaning as pain finder. No filtering, just prep.
# Saves: growth_step4_clean_{RUN_ID}.csv
# ============================================================

STEP3_CSV = csv_name(3, "raw")
STEP4_CSV = csv_name(4, "clean")

df = pd.read_csv(STEP3_CSV)
print(f"Loaded: {len(df):,} rows")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'[#*_~`>]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["title"] = df["title"].apply(clean_text)
df["body"] = df["body"].fillna("").apply(clean_text)
df["top_comments_text"] = df["top_comments_text"].fillna("").apply(clean_text)
df["full_text"] = (df["title"] + " " + df["body"] + " " + df["top_comments_text"]).str.strip()
df["text_length"] = df["full_text"].str.len()
df["has_body"] = (df["body"].str.len() > 10).astype(int)

df.to_csv(STEP4_CSV, index=False)
print(f" CLEANED: {len(df):,} --> {STEP4_CSV}")
print(f"  Avg text length: {df['text_length'].mean():.0f} chars")

In [ ]:
# ============================================================
# STEP 5 -- GROWTH SIGNAL SCORING
# ============================================================
# Instead of pain scoring, we score for growth-tactic content.
# Posts that mention specific numbers, channels, tactics,
# and timelines score highest.
#
# Gate: must contain >= 1 GATE_KEYWORD
# Score: GROWTH_SIGNALS (positive) + EXCLUDE_SIGNALS (negative)
#
# Fields added:
#   growth_score      sum of matched signal weights
#   matched_signals   which signals hit
#   has_numbers       whether post contains dollar/user amounts
#
# Saves: growth_step5_scored_{RUN_ID}.csv
# ============================================================

STEP4_CSV = csv_name(4, "clean")
STEP5_CSV = csv_name(5, "scored")

df = pd.read_csv(STEP4_CSV)
print(f"Loaded: {len(df):,} rows")

ft = df["full_text"].fillna("").str.lower()

# ── Gate filter ──────────────────────────────────────────────
gate = pd.Series(False, index=df.index)
for kw in GATE_KEYWORDS:
    gate |= ft.str.contains(kw, regex=False)
before = len(df)
df = df.loc[gate].copy()
print(f"Gate: {before:,} --> {len(df):,}")

# ── Growth scoring ───────────────────────────────────────────
ft = df["full_text"].fillna("").str.lower()
scores = pd.Series(0.0, index=df.index)
signals = pd.Series("", index=df.index)

for signal, weight in {**GROWTH_SIGNALS, **EXCLUDE_SIGNALS}.items():
    mask = ft.str.contains(signal, regex=False)
    scores += mask.astype(float) * weight
    if weight > 0:
        signals = signals.where(~mask, signals + signal + ", ")

df["growth_score"] = scores.round(1)
df["matched_signals"] = signals.str.rstrip(", ")

# ── Detect specific numbers (strong specificity signal) ──────
# Posts with "$X", "X users", "X MRR" are more actionable
number_pattern = r'(\$[\d,]+[kK]?|\d+[kK]?\s*(users|customers|mrr|arr|revenue|subscribers|month))'
df["has_numbers"] = ft.str.contains(number_pattern, regex=True).astype(int)
df.loc[df["has_numbers"] == 1, "growth_score"] += 15

# ── Cut low scorers ──────────────────────────────────────────
before = len(df)
df = df.loc[df["growth_score"] >= MIN_GROWTH_SCORE].copy()
df = df.sort_values("growth_score", ascending=False).reset_index(drop=True)
print(f"Score floor ({MIN_GROWTH_SCORE}): {before:,} --> {len(df):,}")

df.to_csv(STEP5_CSV, index=False)

print(f"\n{'='*55}")
print(f" GROWTH SCORED: {len(df):,} --> {STEP5_CSV}")
print(f"{'='*55}")
print(f"  Score range: {df['growth_score'].min():.0f} -- {df['growth_score'].max():.0f}")
print(f"  Posts with numbers: {df['has_numbers'].sum()}")
print(f"\n  Top 10:")
for _, r in df.head(10).iterrows():
    print(f"  [{r['growth_score']:5.1f}] {'$' if r['has_numbers'] else ' '} r/{r['subreddit']:15s} {r['title'][:60]}")

In [ ]:
# ============================================================
# STEP 6 -- QUALITY + ENGAGEMENT FILTER
# ============================================================
# Growth posts need higher quality bars than pain posts.
# A 0-upvote post about growth is likely self-promo spam.
# A short body means no tactics shared.
#
# Filters:
#   - MIN_UPVOTES (community validated)
#   - MIN_COMMENTS (people engaged with it)
#   - MIN_BODY_LENGTH (enough text to contain tactics)
#
# Also adds engagement_score and comment_quality.
#
# Saves: growth_step6_quality_{RUN_ID}.csv
# ============================================================

STEP5_CSV = csv_name(5, "scored")
STEP6_CSV = csv_name(6, "quality")

df = pd.read_csv(STEP5_CSV)
print(f"Loaded: {len(df):,} rows")

# ── Engagement scoring ────────────────────────────────────────
def avg_comment_score(s):
    if not isinstance(s, str) or not s.strip():
        return 0.0
    try:
        vals = [float(x.strip()) for x in s.split(",") if x.strip()]
        return np.mean(vals) if vals else 0.0
    except ValueError:
        return 0.0

df["comment_quality"] = df["top_comments_scores"].apply(avg_comment_score)
df["engagement_score"] = (
    np.log1p(df["upvotes"]) * 2.5
    + np.log1p(df["num_comments"]) * 3.5
    + np.log1p(df["comment_quality"]) * 2.0
    + df["upvote_ratio"].fillna(0.5) * 3.0
).round(1)

# ── Quality filters ──────────────────────────────────────────
before = len(df)
df = df.loc[
    (df["upvotes"] >= MIN_UPVOTES)
    & (df["num_comments"] >= MIN_COMMENTS)
    & (df["text_length"] >= MIN_BODY_LENGTH)
].copy()
print(f"Quality filter: {before:,} --> {len(df):,}  (-{before - len(df):,})")

df.to_csv(STEP6_CSV, index=False)

print(f"\n{'='*55}")
print(f" QUALITY FILTERED: {len(df):,} --> {STEP6_CSV}")
print(f"{'='*55}")
print(f"  Engagement range: {df['engagement_score'].min():.1f} -- {df['engagement_score'].max():.1f}")

In [ ]:
# ============================================================
# STEP 7 -- TOPIC EXCLUSIONS + TACTIC TAGGING
# ============================================================
# Removes excluded topics, then tags each post with which
# growth channels/tactics it discusses.
#
# Tactic tags are stored as a comma-separated column so
# you can filter/group by tactic later.
#
# Fields added:
#   tactics_mentioned    comma-sep list of tactics found
#   tactic_count         how many distinct tactics mentioned
#   has_failure_lesson   whether post discusses what DIDN'T work
#
# Saves: growth_step7_tagged_{RUN_ID}.csv
# ============================================================

STEP6_CSV = csv_name(6, "quality")
STEP7_CSV = csv_name(7, "tagged")

df = pd.read_csv(STEP6_CSV)
print(f"Loaded: {len(df):,} rows")

ft = df["full_text"].fillna("").str.lower()

# ── Topic exclusions ─────────────────────────────────────────
topic_mask = pd.Series(False, index=df.index)
for topic in EXCLUDE_TOPICS:
    topic_mask |= ft.str.contains(topic.lower(), regex=False)
before = len(df)
df = df.loc[~topic_mask].copy()
print(f"Topic exclusions: {before:,} --> {len(df):,}")

# ── Title pattern exclusions ─────────────────────────────────
before = len(df)
pat_mask = pd.Series(False, index=df.index)
for pat in EXCLUDE_TITLE_PATTERNS:
    pat_mask |= df["title"].fillna("").str.lower().str.contains(pat, regex=True, na=False)
df = df.loc[~pat_mask].copy()
print(f"Title patterns:   {before:,} --> {len(df):,}")

# ── Tactic tagging ───────────────────────────────────────────
TACTIC_TAGS = {
    "seo":              ["seo", "search engine", "organic traffic", "google ranking", "backlink"],
    "content":          ["content marketing", "blog post", "blogging", "newsletter", "content strategy"],
    "cold_outreach":    ["cold email", "cold outreach", "cold dm", "cold message", "outbound"],
    "paid_ads":         ["facebook ads", "google ads", "paid ads", "ppc", "ad spend", "meta ads"],
    "product_hunt":     ["product hunt", "producthunt"],
    "hacker_news":      ["hacker news", "hackernews", "hn launch"],
    "social_organic":   ["twitter", "x.com", "linkedin", "reddit", "social media"],
    "community":        ["community", "discord", "slack group", "forum", "subreddit"],
    "referral":         ["referral", "word of mouth", "affiliate", "refer a friend"],
    "partnerships":     ["partnership", "integration", "co-marketing", "collab"],
    "lifetime_deal":    ["lifetime deal", "ltd", "appsumo"],
    "freemium":         ["freemium", "free tier", "free plan", "free trial"],
    "pricing":          ["pricing", "price change", "pricing experiment", "raised prices"],
    "onboarding":       ["onboarding", "activation", "first-time user", "welcome email"],
    "retention":        ["churn", "retention", "engagement", "reactivat"],
    "marketplace":      ["marketplace", "app store", "chrome web store", "plugin directory"],
}

ft = df["full_text"].fillna("").str.lower()
tactics_list = []

for _, row_ft in ft.items():
    found = []
    for tactic, keywords in TACTIC_TAGS.items():
        if any(kw in row_ft for kw in keywords):
            found.append(tactic)
    tactics_list.append(", ".join(found))

df["tactics_mentioned"] = tactics_list
df["tactic_count"] = df["tactics_mentioned"].apply(lambda x: len(x.split(", ")) if x else 0)

# ── Failure lessons (extra valuable) ─────────────────────────
failure_patterns = [
    "didn't work", "didn't help", "waste of time", "waste of money",
    "failed", "mistake", "wrong", "wouldn't recommend",
    "don't bother", "stopped doing", "dropped",
    "what i'd do differently", "if i could go back",
]
df["has_failure_lesson"] = ft.apply(
    lambda t: int(any(p in t for p in failure_patterns))
)

df = df.reset_index(drop=True)
df.to_csv(STEP7_CSV, index=False)

print(f"\n{'='*55}")
print(f" TAGGED: {len(df):,} --> {STEP7_CSV}")
print(f"{'='*55}")
print(f"  Posts with tactics: {(df['tactic_count'] > 0).sum()}")
print(f"  Posts with failure lessons: {df['has_failure_lesson'].sum()}")
print(f"\n  Tactic distribution:")
all_tactics = ", ".join(df["tactics_mentioned"].dropna()).split(", ")
all_tactics = [t.strip() for t in all_tactics if t.strip()]
if all_tactics:
    for tactic, count in pd.Series(all_tactics).value_counts().head(15).items():
        print(f"    {tactic:20s} {count:4d}")

In [ ]:
# ============================================================
# STEP 8 -- GROUP BY THEME
# ============================================================
# Groups similar growth posts using TF-IDF (same as pain
# finder), but also enriches each group with:
#   - Which tactics the group covers
#   - Whether it contains failure lessons
#   - Revenue/traction numbers mentioned
#
# Saves: growth_step8_grouped_{RUN_ID}.csv
# ============================================================

STEP7_CSV = csv_name(7, "tagged")
STEP8_CSV = csv_name(8, "grouped")

df = pd.read_csv(STEP7_CSV)
print(f"Loaded: {len(df):,} posts")

titles = df["title"].fillna("").tolist()
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
matrix = tfidf.fit_transform(titles)
sim = cosine_similarity(matrix)

assigned = set()
clusters = []

members_valid = members.dropna(subset=["growth_score"])
if members_valid.empty:
    continue
df = members_valid.loc[members_valid["growth_score"].idxmax()]
order = df["growth_score"].values.argsort()[::-1]

for idx in order:
    if idx in assigned:
        continue
    neighbors = [j for j in range(len(df)) if j not in assigned and sim[idx, j] >= GROUP_SIMILARITY]
    assigned.update(neighbors)
    clusters.append(neighbors)

group_rows = []
for gid, member_idxs in enumerate(clusters):
    members = df.iloc[member_idxs]
    best = members.loc[members["growth_score"].idxmax()]

    # Aggregate tactics across group
    all_tactics = ", ".join(members["tactics_mentioned"].dropna()).split(", ")
    all_tactics = [t.strip() for t in all_tactics if t.strip()]
    tactic_counts = pd.Series(all_tactics).value_counts() if all_tactics else pd.Series(dtype=int)

    sub_dist = members["subreddit"].value_counts()
    top_subs = ", ".join(f"{s}({n})" for s, n in sub_dist.head(4).items())

    combined = "\n".join(f"- {t}" for t in members["title"].tolist())
    if isinstance(best["body"], str) and len(best["body"]) > 20:
        combined += f"\n\nBest post:\n{best['body'][:2000]}"

    group_rows.append({
        "group_id":           gid,
        "post_count":         len(members),
        "post_ids":           ", ".join(members["post_id"].astype(str).tolist()),
        "rep_title":          best["title"],
        "combined_text":      combined[:5000],
        "sample_titles":      " | ".join(members["title"].head(3).tolist())[:300],
        "top_url":            best["url"],
        "total_upvotes":      int(members["upvotes"].sum()),
        "total_comments":     int(members["num_comments"].sum()),
        "avg_growth_score":   round(members["growth_score"].mean(), 1),
        "max_growth_score":   round(members["growth_score"].max(), 1),
        "avg_engagement":     round(members["engagement_score"].mean(), 1),
        "has_numbers":        int(members["has_numbers"].sum()),
        "has_failure_lessons": int(members["has_failure_lesson"].sum()),
        "top_tactics":        ", ".join(tactic_counts.head(5).index.tolist()),
        "tactic_variety":     tactic_counts.nunique() if len(tactic_counts) > 0 else 0,
        "top_subreddits":     top_subs,
    })

df_g = pd.DataFrame(group_rows)

# Merge near-duplicates
if len(df_g) > 1:
    tfidf2 = TfidfVectorizer(max_features=3000, stop_words="english", ngram_range=(1, 2))
    mat2 = tfidf2.fit_transform(df_g["rep_title"].fillna(""))
    sim2 = cosine_similarity(mat2)
    merged = {}
    idxs = df_g.index.tolist()
    for i_pos in range(len(idxs)):
        i = idxs[i_pos]
        if i in merged:
            continue
        for j_pos in range(i_pos + 1, len(idxs)):
            j = idxs[j_pos]
            if j in merged:
                continue
            if sim2[i_pos, j_pos] >= MERGE_THRESHOLD:
                merged[j] = i
    for child, parent in merged.items():
        if parent in df_g.index and child in df_g.index:
            for col in ["post_count", "total_upvotes", "total_comments", "has_numbers", "has_failure_lessons"]:
                df_g.at[parent, col] += df_g.at[child, col]
            df_g.at[parent, "post_ids"] = str(df_g.at[parent, "post_ids"]) + ", " + str(df_g.at[child, "post_ids"])
    df_g = df_g.drop(index=[c for c in merged.keys() if c in df_g.index])

df_g = df_g.reset_index(drop=True)
df_g.to_csv(STEP8_CSV, index=False)

print(f"\n{'='*55}")
print(f" GROUPED: {len(df_g):,} groups --> {STEP8_CSV}")
print(f"{'='*55}")
print(f"  Groups with numbers: {(df_g['has_numbers'] > 0).sum()}")
print(f"  Groups with failure lessons: {(df_g['has_failure_lessons'] > 0).sum()}")

In [ ]:
# ============================================================
# STEP 9 -- FINAL SCORING + EXPORT
# ============================================================
# Ranks growth playbook groups by actionability.
#
# Score weights:
#   Growth signals:   25%  (tactics mentioned)
#   Engagement:       20%  (community validated)
#   Specificity:      25%  (has real numbers)
#   Tactic variety:   15%  (multiple channels discussed)
#   Failure lessons:  15%  (what didn't work = gold)
#
# Saves: growth_step9_final_{RUN_ID}.csv
# ============================================================

STEP8_CSV = csv_name(8, "grouped")
STEP9_CSV = csv_name(9, "final")

df = pd.read_csv(STEP8_CSV)
print(f"Loaded: {len(df):,} groups")

def norm(series, cap_pct=95):
    cap = series.quantile(cap_pct / 100)
    capped = series.clip(upper=cap)
    mn, mx = capped.min(), capped.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return ((capped - mn) / (mx - mn)).round(4)

df["total_engagement"] = df["total_upvotes"] + df["total_comments"]

df["growth_norm"]    = norm(df["avg_growth_score"])
df["eng_norm"]       = norm(np.log1p(df["total_engagement"]))
df["specific_norm"]  = norm(df["has_numbers"])
df["variety_norm"]   = norm(df["tactic_variety"])
df["failure_norm"]   = norm(df["has_failure_lessons"])

df["final_score"] = (
    df["growth_norm"]   * 25
    + df["eng_norm"]    * 20
    + df["specific_norm"] * 25
    + df["variety_norm"]  * 15
    + df["failure_norm"]  * 15
).round(1)

df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
df.to_csv(STEP9_CSV, index=False)

print(f"\n{'='*65}")
print(f" FINAL: {len(df):,} growth playbook groups --> {STEP9_CSV}")
print(f"{'='*65}")
print(f"  Score range: {df['final_score'].min():.1f} -- {df['final_score'].max():.1f}")

print(f"\n  TOP 25 GROWTH PLAYBOOKS:")
for i, g in df.head(25).iterrows():
    nums = 'HAS $' if g['has_numbers'] > 0 else '     '
    fail = 'FAIL' if g['has_failure_lessons'] > 0 else '    '
    print(f"\n  #{i+1} [Score: {g['final_score']}] ({g['post_count']}x) {nums} {fail}")
    print(f"    {g['rep_title'][:80]}")
    print(f"    Tactics: {g['top_tactics'][:60]}")
    print(f"    Subs: {g['top_subreddits'][:50]} | ^{g['total_upvotes']} {g['total_comments']}c")

print(f"\n{'='*65}")
print(f" ALL CSVs:")
for s, l in [(3,"raw"),(4,"clean"),(5,"scored"),(6,"quality"),(7,"tagged"),(8,"grouped"),(9,"final")]:
    f = csv_name(s, l)
    if os.path.exists(f):
        rows = len(pd.read_csv(f))
        print(f"  {f:45s} {rows:>6,} rows")
print(f"{'='*65}")